In [11]:
from __future__ import print_function

import os
import random

import numpy as np
import cv2

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.path import Path

from PIL import Image

import torch

from utils.inpainting_utils import *

imsize = -1
dim_div_by = 64

In [7]:
def get_random_shape(edge_num=9, ratio=0.7, width=432, height=240):
    '''
      edge_num — количество потенциально острых углов
      ratio — величина отклонения от единичной окружности, значение от 0 до 1
      width - ширина изображения
      height - высота изображения

      возвращает случайную фигуру
    '''
    points_num = edge_num*3 + 1
    angles = np.linspace(0, 2*np.pi, points_num)
    codes = np.full(points_num, Path.CURVE4)
    codes[0] = Path.MOVETO
    
    verts = np.stack((np.cos(angles), np.sin(angles))).T * \
        (2*ratio*np.random.random(points_num)+1-ratio)[:, None]
    verts[-1, :] = verts[0, :]
    path = Path(verts, codes)

    fig = plt.figure()
    ax = fig.add_subplot(111)
    patch = patches.PathPatch(path, facecolor='black', lw=2)
    ax.add_patch(patch)
    ax.set_xlim(np.min(verts)*1.1, np.max(verts)*1.1)
    ax.set_ylim(np.min(verts)*1.1, np.max(verts)*1.1)
    ax.axis('off')
    fig.canvas.draw()

    data = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    data = data.reshape((fig.canvas.get_width_height()[::-1] + (4,)))
    data = data[:, :, :3]
    plt.close(fig)
   
    data = cv2.resize(data, (width, height))[:, :, 0]
    data = (1 - np.array(data > 0).astype(np.uint8))*255
    corrdinates = np.where(data > 0)
    xmin, xmax, ymin, ymax = np.min(corrdinates[0]), np.max(
        corrdinates[0]), np.min(corrdinates[1]), np.max(corrdinates[1])
    region = Image.fromarray(data).crop((ymin, xmin, ymax, xmax))
    return region

def create_random_shape_with_random_motion(video_length, imageHeight=240, imageWidth=432):
    '''
      video_length — число кадров в видео
      width - ширина одного кадра
      height - высота одного кадра

      возвращает список масок
    '''
    
    # получаем случайную фигуру
    height = random.randint(imageHeight//3, imageHeight-1)
    width = random.randint(imageWidth//3, imageWidth-1)
    edge_num = random.randint(6, 8)
    ratio = random.randint(6, 8)/10
    region = get_random_shape(
        edge_num=edge_num, ratio=ratio, height=height, width=width)
    region_width, region_height = region.size
    # получаем случайную точку
    x, y = random.randint(
        0, imageHeight-region_height), random.randint(0, imageWidth-region_width)
    m = Image.fromarray(np.zeros((imageHeight, imageWidth)).astype(np.uint8))
    m.paste(region, (y, x, y+region.size[0], x+region.size[1]))
    masks = [m.convert('L')]
    return masks*video_length

In [8]:
dictt = {"bear": 82, "bike-packing": 69, "blackswan": 50, "bmx-bumps": 90, "bmx-trees": 80, "boat": 75, "boxing-fisheye": 87, "breakdance": 84, "breakdance-flare": 71, "bus": 80, "camel": 90, "car-roundabout": 75, "car-shadow": 40, "car-turn": 80, "cat-girl": 89, "classic-car": 63, "color-run": 84, "cows": 104, "crossing": 52, "dance-jump": 60, "dance-twirl": 90, "dancing": 62, "disc-jockey": 76, "dog": 60, "dog-agility": 25, "dog-gooses": 86, "dogs-jump": 66, "dogs-scale": 83, "drift-chicane": 52, "drift-straight": 50, "drift-turn": 64, "drone": 91, "elephant": 80, "flamingo": 80, "goat": 90, "gold-fish": 78, "hike": 80, "hockey": 75, "horsejump-high": 50, "horsejump-low": 60, "india": 81, "judo": 34, "kid-football": 68, "kite-surf": 50, "kite-walk": 80, "koala": 100, "lab-coat": 47, "lady-running": 65, "libby": 49, "lindy-hop": 73, "loading": 50, "longboard": 52, "lucia": 70, "mallard-fly": 70, "mallard-water": 80, "mbike-trick": 79, "miami-surf": 70, "motocross-bumps": 60, "motocross-jump": 40, "motorbike": 43, "night-race": 46, "paragliding": 70, "paragliding-launch": 80, "parkour": 100, "pigs": 79, "planes-water": 38, "rallye": 50, "rhino": 90, "rollerblade": 35, "schoolgirls": 80, "scooter-black": 43, "scooter-board": 91, "scooter-gray": 75, "sheep": 68, "shooting": 40, "skate-park": 80, "snowboard": 66, "soapbox": 99, "soccerball": 48, "stroller": 91, "stunt": 71, "surf": 55, "swing": 60, "tennis": 70, "tractor-sand": 76, "train": 80, "tuk-tuk": 59, "upside-down": 65, "varanus-cage": 67, "walking": 72}

In [14]:
#предобработка всех видео датасета DAVIS с созданием рандомизированной маски

i=1
img_np=None
for name in dictt.keys():
    orig_folder = 'data/DAVIS/'+name
    new_folder = 'data/videos/'+str(i)
    os.mkdir(new_folder)
    for cadr in range(dictt[name]):
        img_path = orig_folder+f'/{cadr:05d}.jpg'
        img_pil, img_np = get_image(img_path, imsize)
        img_pil = crop_image(img_pil, dim_div_by)
        img_np = pil_to_np(img_pil)
        img_var = np_to_torch(img_np)
        textt = f'frame_{cadr:04d}_gt'
        np_to_pil(img_np).save(new_folder+"/"+textt+".png")
    img_mask_pil = create_random_shape_with_random_motion(video_length=1, imageHeight=img_np.shape[1], imageWidth=img_np.shape[2])[0]
    img_mask_np =1 - pil_to_np(img_mask_pil)
    np_to_pil(img_mask_np).save(new_folder+"/mask.png")
    i+=1